# Generate Initial Tennis Ball Coordinates

This notebook runs GridTrackNet from `model_weights.h5`, decodes its 5-frame grid outputs, and exports one compatible coordinate record per source frame.

Each output filename starts with the uploaded video's filename (without its extension). For example, `match01.mp4` produces:
- `match01_ball_coords_raw.json`
- `match01_ball_coords_match_frames.json`
- `match01_ball_coords_raw_overlay.mp4`
- `match01_ball_coords_trajectory.png

In [ ]:
from __future__ import annotations

import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import tensorflow as tf
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in {"scripts", "experiments"} and PROJECT_ROOT.parent.name == "cv":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "cv":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from cv.config import INPUT_DIR, OUTPUT_DIR as CV_OUTPUT_DIR
from cv.models.ball_tracking.gridtracknet import (
    GridTrackNet,
    extract_gridtracknet_coords,
    load_gridtracknet_h5,
)

## 1. Configure Paths

Set `INPUT_VIDEO_PATH` to the uploaded video. Output names are derived automatically from this filename.

In [ ]:
INPUT_VIDEO_PATH = INPUT_DIR / "realtennis.mp4"

# GridTrackNet's published H5 weights. Override this path for another checkpoint.
BALL_MODEL_PATH = Path("/Users/mac/gridtracknet/GridTrackNet/model_weights.h5")

OUTPUT_DIR = CV_OUTPUT_DIR
OUTPUT_PREFIX = INPUT_VIDEO_PATH.stem
RAW_OUTPUT_PATH = OUTPUT_DIR / f"{OUTPUT_PREFIX}_ball_coords_raw.json"
MATCH_FRAMES_OUTPUT_PATH = OUTPUT_DIR / f"{OUTPUT_PREFIX}_ball_coords_match_frames.json"
RAW_POINTS_VIDEO_PATH = OUTPUT_DIR / f"{OUTPUT_PREFIX}_ball_coords_raw_overlay.mp4"
TRAJECTORY_PLOT_PATH = OUTPUT_DIR / f"{OUTPUT_PREFIX}_ball_coords_trajectory.png"

# Set to an integer for a shorter test run, or None for the whole video.
MAX_FRAMES = None

# GridTrackNet returns one confidence grid and two offsets per frame.
GRID_CONFIDENCE_THRESHOLD = 0.5

print("video:", INPUT_VIDEO_PATH)
print("ball model:", BALL_MODEL_PATH)
print("raw output:", RAW_OUTPUT_PATH)
print("match_frames output:", MATCH_FRAMES_OUTPUT_PATH)
print("overlay video:", RAW_POINTS_VIDEO_PATH)
print("trajectory plot:", TRAJECTORY_PLOT_PATH)

In [ ]:
if not INPUT_VIDEO_PATH.exists():
    raise FileNotFoundError(f"Input video not found: {INPUT_VIDEO_PATH}")
if not BALL_MODEL_PATH.exists():
    raise FileNotFoundError(
        "GridTrackNet H5 model not found. Set BALL_MODEL_PATH to model_weights.h5. "
        f"Current value: {BALL_MODEL_PATH}"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load Model

The exported pixel coordinates use the source video's original resolution.

In [ ]:
model = GridTrackNet()
load_gridtracknet_h5(model, BALL_MODEL_PATH)
device = None
print("tensorflow devices:", tf.config.list_physical_devices())

## 3. Extract Initial Coordinates

For every frame, `candidates` contains the highest-confidence GridTrackNet grid cell. `pixel` remains as a backward-compatible alias for that candidate. No temporal path selection, interpolation, or trajectory cleanup is applied.

In [ ]:
_progress_cap = cv2.VideoCapture(str(INPUT_VIDEO_PATH))
_progress_total = int(_progress_cap.get(cv2.CAP_PROP_FRAME_COUNT))
_progress_cap.release()
progress = tqdm(total=MAX_FRAMES or _progress_total, desc="Extract GridTrackNet coordinates")
try:
    ball_coords, video_metadata = extract_gridtracknet_coords(
        INPUT_VIDEO_PATH,
        model,
        device=device,
        max_frames=MAX_FRAMES,
        confidence_threshold=GRID_CONFIDENCE_THRESHOLD,
        progress_callback=lambda current, total: progress.update(1),
    )
finally:
    progress.close()

fps = float(video_metadata["fps"])
frame_width = int(video_metadata["video_width"])
frame_height = int(video_metadata["video_height"])

detected = sum(item["candidate_count"] > 0 for item in ball_coords)
multi_candidate_frames = sum(item["candidate_count"] > 1 for item in ball_coords)
all_candidates = [candidate for item in ball_coords for candidate in item["candidates"]]
valid_confidences = [candidate["confidence"] for candidate in all_candidates]
print(
    f"frames={len(ball_coords):,}, frames_with_candidates={detected:,}, "
    f"total_candidates={len(all_candidates):,}, multi_candidate_frames={multi_candidate_frames:,}"
)


## 4. Plot Coordinate Functions and Trajectory

The left plots show every clustered candidate in `x(t)` and `y(t)`. The right plot shows all candidates in video coordinates, colored by time and sized by confidence. Candidates are not connected across frames because no temporal identity has been selected yet.

In [ ]:
candidate_times = np.array(
    [item["time_sec"] for item in ball_coords for _ in item["candidates"]],
    dtype=np.float64,
)
candidate_pixel_rows = [candidate["pixel"] for item in ball_coords for candidate in item["candidates"]]
candidate_pixels = np.asarray(candidate_pixel_rows, dtype=np.float64).reshape((-1, 2)) if candidate_pixel_rows else np.empty((0, 2), dtype=np.float64)
candidate_confidences = np.array(
    [candidate["confidence"] for item in ball_coords for candidate in item["candidates"]],
    dtype=np.float64,
)

fig = plt.figure(figsize=(15, 8), constrained_layout=True)
grid = fig.add_gridspec(2, 2, width_ratios=(1.5, 1.0))
ax_x = fig.add_subplot(grid[0, 0])
ax_y = fig.add_subplot(grid[1, 0], sharex=ax_x)
ax_candidates = fig.add_subplot(grid[:, 1])

if len(candidate_pixels):
    marker_sizes = 10.0 + 38.0 * candidate_confidences
    ax_x.scatter(candidate_times, candidate_pixels[:, 0], c=candidate_confidences, cmap="plasma", s=8, alpha=0.75)
    ax_y.scatter(candidate_times, candidate_pixels[:, 1], c=candidate_confidences, cmap="plasma", s=8, alpha=0.75)
    points = ax_candidates.scatter(
        candidate_pixels[:, 0],
        candidate_pixels[:, 1],
        c=candidate_times,
        cmap="viridis",
        s=marker_sizes,
        alpha=0.75,
        edgecolors="none",
    )
    fig.colorbar(points, ax=ax_candidates, label="time (s)")
else:
    ax_candidates.text(0.5, 0.5, "No candidates detected", ha="center", va="center", transform=ax_candidates.transAxes)

ax_x.set_ylabel("x (pixel)")
ax_y.set_ylabel("y (pixel)")
ax_y.set_xlabel("time (s)")
ax_x.set_title("All candidate x coordinates")
ax_y.set_title("All candidate y coordinates")
ax_x.grid(alpha=0.25)
ax_y.grid(alpha=0.25)

ax_candidates.set_xlim(0, frame_width)
ax_candidates.set_ylim(frame_height, 0)
ax_candidates.set_aspect("equal", adjustable="box")
ax_candidates.set_xlabel("x (pixel)")
ax_candidates.set_ylabel("y (pixel)")
ax_candidates.set_title("All clustered candidates in video coordinates")
ax_candidates.grid(alpha=0.2)

fig.suptitle(f"GridTrackNet coordinates: {INPUT_VIDEO_PATH.name}", fontsize=14)
fig.savefig(TRAJECTORY_PLOT_PATH, dpi=160, bbox_inches="tight")
print("wrote candidate plot:", TRAJECTORY_PLOT_PATH)
plt.show()

## 5. Render Initial Coordinate Video

This video draws the GridTrackNet candidate in each frame. Marker color and label show confidence; no temporal tail is drawn because candidates have not been associated across frames.

In [ ]:
def render_initial_coordinate_overlay(
    input_video_path,
    output_video_path,
    ball_coords,
    *,
    fps,
    frame_width,
    frame_height,
    ball_radius=7,
):
    cap = cv2.VideoCapture(str(input_video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video for rendering: {input_video_path}")

    output_video_path = Path(output_video_path)
    output_video_path.parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(
        str(output_video_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps if fps > 0 else 25.0,
        (frame_width, frame_height),
    )
    if not writer.isOpened():
        cap.release()
        raise RuntimeError(f"Could not open video writer: {output_video_path}")

    candidates_by_frame = {item["frame_index"]: item["candidates"] for item in ball_coords}
    total_count = len(ball_coords)
    detected_count = sum(bool(candidates) for candidates in candidates_by_frame.values())
    total_candidates = sum(len(candidates) for candidates in candidates_by_frame.values())
    rendered_count = 0
    progress = tqdm(total=total_count, desc="Render initial coordinate overlay")

    for frame_index in range(total_count):
        ok, frame = cap.read()
        if not ok:
            break
        canvas = frame.copy()

        frame_candidates = candidates_by_frame.get(frame_index, [])
        for candidate in frame_candidates:
            x, y = (int(round(value)) for value in candidate["pixel"])
            confidence = float(candidate["confidence"])
            color = (0, int(round(255 * confidence)), int(round(255 * (1.0 - confidence))))
            cv2.circle(canvas, (x, y), ball_radius + 3, (0, 0, 0), -1, cv2.LINE_AA)
            cv2.circle(canvas, (x, y), ball_radius, color, -1, cv2.LINE_AA)
            cv2.drawMarker(canvas, (x, y), (255, 255, 255), cv2.MARKER_CROSS, 12, 1, cv2.LINE_AA)
            label = f"#{candidate['rank']} {confidence:.3f}"
            label_x = min(max(4, x + ball_radius + 5), max(4, frame_width - 120))
            label_y = min(max(18, y - ball_radius - 4), frame_height - 6)
            cv2.putText(canvas, label, (label_x, label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.46, (0, 0, 0), 3, cv2.LINE_AA)
            cv2.putText(canvas, label, (label_x, label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.46, color, 1, cv2.LINE_AA)

        hud_lines = [
            f"frame {frame_index}/{total_count - 1}",
            f"candidates this frame: {len(frame_candidates)}",
            f"frames with candidates: {detected_count}/{total_count} ({detected_count / max(total_count, 1):.1%})",
            f"total candidates: {total_candidates}",
        ]
        for line_index, text in enumerate(hud_lines):
            y_pos = 32 + line_index * 26
            cv2.putText(canvas, text, (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (0, 0, 0), 4, cv2.LINE_AA)
            cv2.putText(canvas, text, (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (255, 255, 255), 1, cv2.LINE_AA)

        writer.write(canvas)
        rendered_count += 1
        progress.update(1)

    cap.release()
    writer.release()
    progress.close()
    return rendered_count


rendered_frames = render_initial_coordinate_overlay(
    INPUT_VIDEO_PATH,
    RAW_POINTS_VIDEO_PATH,
    ball_coords,
    fps=fps,
    frame_width=frame_width,
    frame_height=frame_height,
)
print(f"wrote overlay video: {RAW_POINTS_VIDEO_PATH} ({rendered_frames} frames)")

## 6. Write JSON Outputs

In [ ]:
metadata = {
    "version": "gridtracknet-ball-candidates-1.0.0",
    "source_video": INPUT_VIDEO_PATH.name,
    "source_video_path": str(INPUT_VIDEO_PATH),
    "output_prefix": OUTPUT_PREFIX,
    "ball_model_path": str(BALL_MODEL_PATH),
    "coordinate_method": "gridtracknet_grid_offset",
    "confidence_definition": "maximum GridTrackNet confidence cell per frame",
    "confidence_threshold": GRID_CONFIDENCE_THRESHOLD,
    "input_resolution": {"width": 768, "height": 432},
    "output_grid": {"columns": 48, "rows": 27},
    "sampling": video_metadata["sampling"],
    "fps": fps,
    "video_width": frame_width,
    "video_height": frame_height,
    "total_frames": len(ball_coords),
    "generated_at": datetime.now(timezone.utc).isoformat(),
}

raw_payload = {
    "metadata": metadata,
    "ball_coords": ball_coords,
    "summary": {
        "detected_frames": detected,
        "missing_frames": len(ball_coords) - detected,
        "detection_ratio": round(detected / max(len(ball_coords), 1), 4),
        "coordinate_method": "gridtracknet_grid_offset",
        "total_candidates": len(all_candidates),
        "multi_candidate_frames": multi_candidate_frames,
        "mean_candidates_per_detected_frame": round(len(all_candidates) / max(detected, 1), 4),
        "confidence_mean": round(float(np.mean(valid_confidences)), 4) if valid_confidences else None,
    },
}


def ball_payload_from_coord_item(item):
    if item["pixel"] is None:
        return None
    return {
        "pixel": item["pixel"],
        "court": None,
        "confidence": item["confidence"],
        "coordinate_method": item["coordinate_method"],
    }


def candidate_payloads_from_coord_item(item):
    return [
        {
            **candidate,
            "court": None,
            "coordinate_method": item["coordinate_method"],
        }
        for candidate in item["candidates"]
    ]


match_frames_payload = {
    "metadata": {
        **metadata,
        "version": "2.1.0-gridtracknet-ball-candidates",
        "court_dimensions": {"unit": "meters", "width": 10.97, "length": 23.77},
    },
    "frames": [
        {
            "frame_index": item["frame_index"],
            "time_sec": item["time_sec"],
            "court_visible": False,
            "court_keypoints_detected": 0,
            "ball": ball_payload_from_coord_item(item),
            "ball_candidates": candidate_payloads_from_coord_item(item),
            "players": [],
            "events": {
                "is_bounce": False,
                "is_hit": False,
                "is_net_hit": False,
            },
        }
        for item in ball_coords
    ],
}

RAW_OUTPUT_PATH.write_text(json.dumps(raw_payload, ensure_ascii=False, indent=2), encoding="utf-8")
MATCH_FRAMES_OUTPUT_PATH.write_text(
    json.dumps(match_frames_payload, ensure_ascii=False, separators=(",", ":")),
    encoding="utf-8",
)

print("wrote:", RAW_OUTPUT_PATH, f"({RAW_OUTPUT_PATH.stat().st_size / 1024:.1f} KB)")
print("wrote:", MATCH_FRAMES_OUTPUT_PATH, f"({MATCH_FRAMES_OUTPUT_PATH.stat().st_size / 1024:.1f} KB)")


## 7. Feed Into Trajectory Lab

After running the notebook, use the generated path directly:

```python
MATCH_FRAMES_PATH = CV_OUTPUT_DIR / f"{INPUT_VIDEO_PATH.stem}_ball_coords_match_frames.json"
```